# Cell Type Embeddings — Data Download & Preprocessing (based on Notebook 1)
## scRNA-seq Finetuning Pipeline

**Dataset**: NeurIPS 2021 10x Multiome Bone Marrow (same as scooby paper)
- BAM files from SRA accession **SRP356158**
- Preprocessed count matrix from GEO accession **GSE194122**

**This notebook produces** (saved to Google Drive):
| File | Description |
|------|-------------|
| `cell_embeddings_aligned.parquet` | scVI 14-dim latent vectors, one row per cell |
| `rna_coverage_aligned.h5ad` | SnapATAC2 per-cell coverage AnnData |
| `gene_intervals_splits.parquet` | Gene → 1 Mb window + train/val/test label |
| `leaking_gene_mask.npy` | Boolean mask of genes in val/test regions |
| `aligned_barcodes.npy` | Ordered cell barcodes |
| `preprocessing_manifest.json` | Summary manifest loaded by Notebook 2 |

---
### ⚠️ Critical implementation notes
1. **Leakage prevention**: `leaking_gene_mask` is applied to `adata` **before** `scvi.model.SCVI.setup_anndata` — never after.
2. **SnapATAC2 fork**: use `lauradmartens/SnapATAC2@scooby`, NOT the PyPI version.
3. **Genome**: everything is hg38 throughout.
4. **BAM size**: Full SRP356158 is ~hundreds of GB. Cell 3 downloads a single prototype run; see inline comments for full download instructions.
5. **API key**: Store your AlphaGenome key as a Colab Secret named `ALPHAGENOME_API_KEY`.

## 0 — Mount Google Drive

In [3]:
# Setup
import sys
print(sys.executable)

/Users/rohankrishnamurthi/Downloads/MIT HST .506/Final Project Code/genai4bio-project/.venv/bin/python


In [3]:
import google.colab
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [8]:


import os

# ── Persistent root on your Drive ──────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position'
DATA_DIR   = f'{DRIVE_ROOT}/data/neurips'
BAM_DIR    = f'{DRIVE_ROOT}/data/bam'
SRA_DIR    = f'{DRIVE_ROOT}/data/sra'
CACHE_DIR  = f'{DRIVE_ROOT}/data/embedding_cache'

for d in [DATA_DIR, BAM_DIR, SRA_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Directories ready:')
for d in [DATA_DIR, BAM_DIR, SRA_DIR, CACHE_DIR]:
    print(' ', d)

Drive mounted. Directories ready:
  /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/neurips
  /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/bam
  /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/sra
  /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/embedding_cache


## 1 — Install Dependencies

In [ ]:
# SnapATAC2 fork from scooby paper — handles scRNA-seq split reads
# IMPORTANT: do NOT install the standard PyPI snapatac2; it cannot handle split reads
!pip install snapatac2-scooby # then restart


In [9]:
!pip install -q alphagenome scvi-tools scanpy anndata pysam pyranges scrublet gdown


print('\nAll packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 29.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached hmmlearn-0.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
  Using cached cykhash-2.0.1-cp312-cp312-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 6.2 MB/s eta 0:00:00
Using cached hmmlearn-0.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (165 kB)
  Created wheel for macs3: filename=macs3-3.0.4-cp312

## 2 — Download Preprocessed Count Matrix (GEO: GSE194122)

In [10]:
import subprocess, os

# ── GEO FTP path for the NeurIPS 2021 multiome dataset ─────────────────────
GEO_URL = (
    'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE194nnn/GSE194122/suppl/'
    'GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz'
)
H5AD_GZ  = f'{DATA_DIR}/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz'
H5AD     = H5AD_GZ.replace('.gz', '')

if os.path.exists(H5AD):
    print(f'Already downloaded: {H5AD}')
else:
    print('Downloading from GEO (~2 GB) ...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', H5AD_GZ, GEO_URL], check=True)
    print('Decompressing ...')
    subprocess.run(['gunzip', '-f', H5AD_GZ], check=True)
    print(f'Saved to: {H5AD}')

print(f'File size: {os.path.getsize(H5AD)/1e9:.2f} GB')

Decompressing ...
Saved to: /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/neurips/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad
File size: 3.12 GB


## 3 — Download scRNA-seq BAM Files (SRA: SRP356158)

In [11]:
import subprocess, os, glob

# ── Install SRA Toolkit ─────────────────────────────────────────────────────
SRA_TK_URL = 'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz'
SRA_TK_TGZ = '/content/sratoolkit.tar.gz'

if not glob.glob('/content/sratoolkit*/bin/prefetch'):
    print('Downloading SRA Toolkit ...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', SRA_TK_TGZ, SRA_TK_URL], check=True)
    subprocess.run(['tar', '-xzf', SRA_TK_TGZ, '-C', '/content/'], check=True)

SRA_BIN = glob.glob('/content/sratoolkit*/bin')[0]
PREFETCH    = f'{SRA_BIN}/prefetch'
FASTERQDUMP = f'{SRA_BIN}/fasterq-dump'
print(f'SRA Toolkit ready: {SRA_BIN}')

SRA Toolkit ready: /content/sratoolkit.3.4.1-ubuntu64/bin


In [12]:
# ── SRA accessions for SRP356158 ─────────────────────────────────────────────
# Full list: https://www.ncbi.nlm.nih.gov/sra/?term=SRP356158
# The NeurIPS multiome dataset has multiple RNA-seq runs per sample.
# For prototyping, we download ONE run; for full training, iterate over all accessions.
#
# Known run accessions from SRP356158 (RNA-seq, 10x Chromium):
#   SRR17010514  s1d1   donor 1, site 1, day 1
#   SRR17010515  s1d2
#   SRR17010516  s1d3
#   SRR17010517  s2d1   donor 1, site 2
#   SRR17010518  s2d4
#   SRR17010519  s2d9
#   SRR17010520  s3d1
#   SRR17010521  s3d6
#   SRR17010522  s3d7
#   SRR17010523  s4d1
#
# Set PROTOTYPE_MODE=True to download only one run for testing;
# set to False to download all runs (warning: hundreds of GB).

PROTOTYPE_MODE = True  # ← change to False for full training run

ALL_RUNS = [
    'SRR17010514', 'SRR17010515', 'SRR17010516', 'SRR17010517',
    'SRR17010518', 'SRR17010519', 'SRR17010520', 'SRR17010521',
    'SRR17010522', 'SRR17010523',
]

RUNS_TO_DOWNLOAD = ALL_RUNS[:1] if PROTOTYPE_MODE else ALL_RUNS
print(f'Will download {len(RUNS_TO_DOWNLOAD)} run(s): {RUNS_TO_DOWNLOAD}')

for run in RUNS_TO_DOWNLOAD:
    run_sra  = f'{SRA_DIR}/{run}/{run}.sra'
    run_fastq_r1 = f'{DATA_DIR}/fastq/{run}_1.fastq'

    if os.path.exists(run_fastq_r1):
        print(f'  [{run}] FASTQ already exists, skipping.')
        continue

    if not os.path.exists(run_sra):
        print(f'  [{run}] Prefetching SRA ...')
        subprocess.run(
            [PREFETCH, run, '--output-directory', SRA_DIR, '--max-size', '100G'],
            check=True
        )

    print(f'  [{run}] Dumping to FASTQ ...')
    os.makedirs(f'{DATA_DIR}/fastq', exist_ok=True)
    subprocess.run(
        [FASTERQDUMP, run_sra, '--outdir', f'{DATA_DIR}/fastq',
         '--split-files', '--threads', '4'],
        check=True
    )

print('Download complete.')

Will download 1 run(s): ['SRR17010514']
  [SRR17010514] Prefetching SRA ...
  [SRR17010514] Dumping to FASTQ ...
Download complete.


In [13]:
# ── Cell 3b: Align FASTQ → BAM with Cell Ranger ──────────────────────────────
#
# SKIP THIS CELL if BAMs are already available (e.g. if SRA provides BAM directly).
# Cell Ranger reference: refdata-gex-GRCh38-2020-A
# Download from: https://www.10xgenomics.com/support/software/cell-ranger/downloads
#
# Install Cell Ranger (requires manual download from 10x Genomics):
#   !wget -O /content/cellranger-8.0.0.tar.gz <YOUR_SIGNED_URL>
#   !tar -xzf /content/cellranger-8.0.0.tar.gz -C /content/
#   !export PATH=/content/cellranger-8.0.0:$PATH
#
# Then run per sample:

CELLRANGER_REF = '/content/drive/MyDrive/project/references/refdata-gex-GRCh38-2020-A'  # adjust path
SAMPLE_ID      = 'SRR17010514'   # one sample at a time

SKIP_CELLRANGER = True  # ← set False when you have Cell Ranger installed

if not SKIP_CELLRANGER:
    subprocess.run([
        'cellranger', 'count',
        f'--id={SAMPLE_ID}',
        f'--fastqs={DATA_DIR}/fastq',
        f'--sample={SAMPLE_ID}',
        f'--transcriptome={CELLRANGER_REF}',
        '--localcores=8',
        '--localmem=64',
        f'--output-dir={BAM_DIR}',
    ], check=True)

    # Expected output: {BAM_DIR}/{SAMPLE_ID}/outs/possorted_genome_bam.bam
    BAM_FILE = f'{BAM_DIR}/{SAMPLE_ID}/outs/possorted_genome_bam.bam'
    print(f'BAM ready: {BAM_FILE}')
else:
    # Use pre-existing BAM (adjust path if you already have one on Drive)
    BAM_FILE = f'{BAM_DIR}/possorted_genome_bam.bam'
    if os.path.exists(BAM_FILE):
        print(f'Using existing BAM: {BAM_FILE}')
    else:
        print('⚠ No BAM found. Set SKIP_CELLRANGER=False or place BAM at:', BAM_FILE)

⚠ No BAM found. Set SKIP_CELLRANGER=False or place BAM at: /content/drive/MyDrive/2026 Spring/MIT - HST 506- GenAI in Bio/project/test_per_position/data/bam/possorted_genome_bam.bam


## 4 — Load and QC the Preprocessed Count Matrix

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd

H5AD = f'{DATA_DIR}/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad'

print('Loading AnnData ...')
adata = ad.read_h5ad(H5AD)
print(adata)
# Expected: ~63,683 cells × ~13,953 genes

# The h5ad may contain both RNA and ATAC modalities; keep only cells with annotations
cell_type_col = 'l2_cell_type' if 'l2_cell_type' in adata.obs.columns else adata.obs.columns[0]
print(f'\nUsing cell type column: {cell_type_col!r}')
print(adata.obs[cell_type_col].value_counts().head(20))

adata = adata[adata.obs[cell_type_col].notna()].copy()
print(f'\nCells after filtering: {adata.n_obs}')

In [ ]:
import scrublet as scr

print('Running Scrublet doublet detection ...')
# Scrublet expects raw counts — use adata.X directly (should be raw counts in this h5ad)
# If adata.X is normalized, reload from adata.raw.to_adata() or the h5ad layer
raw_matrix = adata.X if not hasattr(adata, 'raw') or adata.raw is None else adata.raw[:, adata.var_names].X

scrub = scr.Scrublet(raw_matrix, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, min_cells=3)

adata.obs['doublet_score'] = doublet_scores
adata.obs['predicted_doublet'] = predicted_doublets

print(f'Predicted doublets: {predicted_doublets.sum()} ({predicted_doublets.mean()*100:.1f}%)')
adata = adata[~adata.obs['predicted_doublet']].copy()
print(f'Cells after doublet removal: {adata.n_obs}')

## 5 — Genomic Train / Val / Test Split & Leakage Masking

> **Why this matters**: genes in val/test genomic windows must be masked from the count matrix *before* fitting scVI. If not, the cell embeddings will encode information about held-out regions, leaking target signal into validation/test evaluation.

In [ ]:
from alphagenome.data import fold_intervals
import pyranges as pr

# Load AlphaGenome fold intervals (identical to Borzoi fold splits)
val_intervals  = fold_intervals.get_fold_intervals(subset=fold_intervals.Subset.VALID)
test_intervals = fold_intervals.get_fold_intervals(subset=fold_intervals.Subset.TEST)
print(f'Val intervals:  {len(val_intervals)}')
print(f'Test intervals: {len(test_intervals)}')

def intervals_to_pyranges(intervals):
    records = [{'Chromosome': iv.chromosome, 'Start': iv.start, 'End': iv.end}
               for iv in intervals]
    return pr.PyRanges(pd.DataFrame(records))

val_pr      = intervals_to_pyranges(val_intervals)
test_pr     = intervals_to_pyranges(test_intervals)
held_out_pr = pr.concat([val_pr, test_pr])
print('PyRanges created for held-out regions.')

In [ ]:
# Load GENCODE v46 annotation (feather format from AlphaGenome GCS bucket)
GTF_FEATHER_URL = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)
GTF_FEATHER_LOCAL = f'{DATA_DIR}/gencode.v46.annotation.gtf.gz.feather'

if not os.path.exists(GTF_FEATHER_LOCAL):
    print('Downloading GENCODE v46 feather ...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', GTF_FEATHER_LOCAL, GTF_FEATHER_URL], check=True)

gtf = pd.read_feather(GTF_FEATHER_LOCAL)
print(f'GTF rows: {len(gtf):,}')
print(gtf[gtf['feature'] == 'gene'].head(3)[['seqname','start','end','gene_name','feature']])

In [ ]:
# Build gene-level PyRanges and find overlap with held-out intervals
genes = (gtf[gtf['feature'] == 'gene']
         [['seqname', 'start', 'end', 'gene_name', 'gene_id']].copy()
         .rename(columns={'seqname': 'Chromosome', 'start': 'Start', 'end': 'End'}))
genes_pr = pr.PyRanges(genes)

overlapping       = genes_pr.overlap(held_out_pr)
held_out_gene_ids  = set(overlapping.df['gene_id'].dropna().tolist())
held_out_gene_names = set(overlapping.df['gene_name'].dropna().tolist())

print(f'Genes overlapping val/test regions: {len(held_out_gene_names)}')

# Create mask — works whether var_names are ENSEMBL IDs or gene symbols
if adata.var_names[0].startswith('ENSG'):
    mask_leaking = adata.var_names.isin(held_out_gene_ids)
else:
    mask_leaking = adata.var_names.isin(held_out_gene_names)

print(f'Leaking genes in AnnData: {mask_leaking.sum()} / {len(mask_leaking)}')

# Save masks for reference
np.save(f'{DATA_DIR}/leaking_gene_mask.npy', mask_leaking)
np.save(f'{DATA_DIR}/held_out_gene_names.npy', np.array(list(held_out_gene_names)))
print('Masks saved.')

## 6 — Train scVI Cell Embedder (on Masked Count Matrix)

> Leakage prevention: the masked AnnData is passed to `setup_anndata` — genes in val/test regions are never seen by scVI.

In [ ]:
import scvi

# ── Apply leakage mask BEFORE setup_anndata — this is the critical step ──────
adata_masked = adata[:, ~mask_leaking].copy()
print(f'Genes for scVI (after masking): {adata_masked.n_vars}')

# scVI needs raw integer counts in .X
# If .X is normalised, restore raw counts
if hasattr(adata, 'raw') and adata.raw is not None:
    adata_masked.X = adata.raw[:, ~mask_leaking].X.copy()
    print('Restored raw counts from adata.raw')
else:
    print('Using adata.X as raw counts (verify these are integer counts)')

batch_key = 'sample' if 'sample' in adata_masked.obs.columns else None
print(f'Batch key: {batch_key}')

scvi.model.SCVI.setup_anndata(
    adata_masked,
    layer=None,         # use .X (raw counts)
    batch_key=batch_key,
)

scvi_model = scvi.model.SCVI(
    adata_masked,
    n_latent=14,           # match scooby's 14-dimensional embedding
    n_layers=2,
    n_hidden=128,
    gene_likelihood='zinb',
)

print(scvi_model)

In [ ]:
# Check if a trained model already exists on Drive (resuming after disconnection)
SCVI_MODEL_DIR = f'{DATA_DIR}/scvi_model'

if os.path.exists(SCVI_MODEL_DIR):
    print(f'Loading existing scVI model from {SCVI_MODEL_DIR} ...')
    scvi_model = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata_masked)
else:
    print('Training scVI (400 epochs, early stopping) ...')
    scvi_model.train(
        max_epochs=400,
        early_stopping=True,
        early_stopping_patience=20,
        batch_size=256,
        plan_kwargs={'lr': 1e-3},
    )
    scvi_model.save(SCVI_MODEL_DIR, overwrite=True)
    print(f'scVI model saved to {SCVI_MODEL_DIR}')

# Plot ELBO (skipped on reload)
try:
    import matplotlib.pyplot as plt
    scvi_model.history['elbo_train'].plot(title='scVI ELBO (train)')
    plt.savefig(f'{DATA_DIR}/scvi_elbo.png', dpi=120, bbox_inches='tight')
    plt.show()
except Exception:
    pass

In [ ]:
import matplotlib.pyplot as plt

# Extract 14-dim latent vectors
cell_embeddings = scvi_model.get_latent_representation()  # (n_cells, 14)
print(f'Cell embedding shape: {cell_embeddings.shape}')

# Save with barcodes + cell-type annotations
embedding_df = pd.DataFrame(
    cell_embeddings,
    index=adata_masked.obs_names,
    columns=[f'z_{i}' for i in range(14)]
)
embedding_df['cell_type'] = adata_masked.obs[cell_type_col].values
embedding_df['batch'] = adata_masked.obs.get('sample', pd.Series(['unknown']*len(adata_masked))).values
embedding_df.to_parquet(f'{DATA_DIR}/cell_embeddings_scvi_14dim.parquet')
print(f'Saved: {DATA_DIR}/cell_embeddings_scvi_14dim.parquet')

# Sanity-check UMAP
adata_masked.obsm['X_scVI'] = cell_embeddings
sc.pp.neighbors(adata_masked, use_rep='X_scVI', n_neighbors=15)
sc.tl.umap(adata_masked)
sc.pl.umap(adata_masked, color=cell_type_col,
           legend_loc='on data', title='scVI embeddings — cell type', frameon=False)
plt.savefig(f'{DATA_DIR}/umap_scvi_celltype.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Convert BAM to Per-cell Coverage (SnapATAC2 scooby fork)

In [ ]:
import snapatac2 as snap

# Export valid barcodes (those surviving QC)
BARCODES_FILE = f'{DATA_DIR}/valid_barcodes.txt'
with open(BARCODES_FILE, 'w') as f:
    for bc in adata.obs_names:
        f.write(bc + '\n')
print(f'Exported {adata.n_obs} valid barcodes → {BARCODES_FILE}')

# ── Verify BAM_FILE exists ────────────────────────────────────────────────────
BAM_FILE = f'{BAM_DIR}/possorted_genome_bam.bam'  # adjust if Cell Ranger placed it differently
if not os.path.exists(BAM_FILE):
    raise FileNotFoundError(
        f'BAM not found: {BAM_FILE}\n'
        'Run Cell 3b (Cell Ranger alignment) first, or update BAM_FILE to the correct path.'
    )

COVERAGE_H5AD = f'{DATA_DIR}/rna_coverage.h5ad'

if os.path.exists(COVERAGE_H5AD):
    print(f'Coverage h5ad already exists: {COVERAGE_H5AD}')
else:
    print('Converting BAM → per-cell coverage fragments (this may take 30-90 min) ...')
    # Key scRNA-seq parameters:
    #   is_paired=False   — scRNA reads are NOT paired in the ATAC sense
    #   shift_left/right=0 — no Tn5 shift correction
    #   xf_filter=True    — keep only Cell Ranger-validated reads (xf:i:25 flag)
    snap.pp.make_fragment_file(
        bam_file=BAM_FILE,
        output_file=COVERAGE_H5AD,
        is_paired=False,
        shift_left=0,
        shift_right=0,
        barcode_tag='CB',
        umi_tag='UB',
        xf_filter=True,
        whitelist=BARCODES_FILE,
        genome='hg38',
        min_num_fragments=0,
        chunk_size=10_000,
    )
    print(f'Saved: {COVERAGE_H5AD}')

In [ ]:
SNAP_H5AD = f'{DATA_DIR}/rna_snapatac.h5ad'

rna_data = snap.pp.import_data(
    fragment_file=COVERAGE_H5AD,
    genome=snap.genome.hg38,
    min_num_fragments=0,
    whitelist=BARCODES_FILE,
    file=SNAP_H5AD,
)
print(rna_data)
# Each row = cell; columns = genomic positions
# Values = signed read lengths (negative = minus strand)

In [ ]:
# Align barcodes between coverage AnnData and embedding DataFrame
common_barcodes = sorted(
    set(rna_data.obs_names) & set(embedding_df.index)
)
print(f'Common barcodes between coverage and embeddings: {len(common_barcodes)}')

rna_data_aligned  = rna_data[common_barcodes].copy()
embeddings_aligned = embedding_df.loc[common_barcodes]

# Save aligned files
np.save(f'{DATA_DIR}/aligned_barcodes.npy', np.array(common_barcodes))
embeddings_aligned.to_parquet(f'{DATA_DIR}/cell_embeddings_aligned.parquet')
rna_data_aligned.write_h5ad(f'{DATA_DIR}/rna_coverage_aligned.h5ad')

print(f'Aligned barcodes : {DATA_DIR}/aligned_barcodes.npy')
print(f'Aligned embeddings: {DATA_DIR}/cell_embeddings_aligned.parquet')
print(f'Aligned coverage  : {DATA_DIR}/rna_coverage_aligned.h5ad')

## 8 — Build Gene-Interval Lookup Table (Train / Val / Test)

In [ ]:
from alphagenome.data import gene_annotation, genome
from alphagenome.models import dna_client

# Filter GTF to protein-coding, MANE Select transcripts
gtf_protein = gene_annotation.filter_protein_coding(gtf)
gtf_mane    = gene_annotation.filter_to_mane_select_transcript(gtf_protein)

# Build 1 Mb windows centered on each gene
HALF = dna_client.SEQUENCE_LENGTH_1MB // 2

gene_records = []
for _, row in gtf_mane[gtf_mane['feature'] == 'gene'].iterrows():
    chrom  = row['seqname']
    start  = int(row['start'])
    end    = int(row['end'])
    center = (start + end) // 2
    iv_start = max(0, center - HALF)
    iv_end   = iv_start + dna_client.SEQUENCE_LENGTH_1MB

    gene_records.append({
        'gene_name':      row.get('gene_name'),
        'gene_id':        row.get('gene_id'),
        'chromosome':     chrom,
        'gene_start':     start,
        'gene_end':       end,
        'strand':         row.get('strand', '.'),
        'interval_start': iv_start,
        'interval_end':   iv_end,
    })

gene_df = pd.DataFrame(gene_records)
print(f'Total genes: {len(gene_df)}')

# Assign train / val / test split based on genomic overlap
def assign_split(row, val_pr, test_pr):
    gpr = pr.PyRanges(pd.DataFrame([{
        'Chromosome': row['chromosome'],
        'Start': row['gene_start'],
        'End': row['gene_end'],
    }]))
    if len(gpr.overlap(test_pr)) > 0:
        return 'test'
    elif len(gpr.overlap(val_pr)) > 0:
        return 'val'
    return 'train'

gene_df['split'] = gene_df.apply(assign_split, axis=1, args=(val_pr, test_pr))

print(gene_df['split'].value_counts())
gene_df.to_parquet(f'{DATA_DIR}/gene_intervals_splits.parquet')
print(f'Saved: {DATA_DIR}/gene_intervals_splits.parquet')

In [ ]:
# ── Build exon coordinate table (used for exon-masked decoding in Notebook 2) ─
exon_records = []
for _, row in gtf_mane[gtf_mane['feature'] == 'exon'].iterrows():
    exon_records.append({
        'gene_id':    row.get('gene_id'),
        'gene_name':  row.get('gene_name'),
        'chromosome': row['seqname'],
        'exon_start': int(row['start']),
        'exon_end':   int(row['end']),
        'strand':     row.get('strand', '.'),
    })

exon_df = pd.DataFrame(exon_records)
exon_df.to_parquet(f'{DATA_DIR}/exon_coordinates.parquet')
print(f'Exon records: {len(exon_df)}')
print(f'Saved: {DATA_DIR}/exon_coordinates.parquet')

## 9 — Save Preprocessing Manifest

In [ ]:
import json
from alphagenome.models import dna_client

manifest = {
    'n_cells':                      len(common_barcodes),
    'n_train_genes':                int((gene_df['split'] == 'train').sum()),
    'n_val_genes':                  int((gene_df['split'] == 'val').sum()),
    'n_test_genes':                 int((gene_df['split'] == 'test').sum()),
    'n_leaking_genes_masked_from_scvi': int(mask_leaking.sum()),
    'embedding_dim':                14,
    'resolution_bp':                1,
    'sequence_length':              dna_client.SEQUENCE_LENGTH_1MB,
    'cell_type_column':             cell_type_col,
    'files': {
        'cell_embeddings':   f'{DATA_DIR}/cell_embeddings_aligned.parquet',
        'rna_coverage':      f'{DATA_DIR}/rna_coverage_aligned.h5ad',
        'gene_intervals':    f'{DATA_DIR}/gene_intervals_splits.parquet',
        'exon_coordinates':  f'{DATA_DIR}/exon_coordinates.parquet',
        'scvi_model':        f'{DATA_DIR}/scvi_model/',
        'barcodes':          f'{DATA_DIR}/aligned_barcodes.npy',
        'leaking_gene_mask': f'{DATA_DIR}/leaking_gene_mask.npy',
    }
}

MANIFEST_PATH = f'{DATA_DIR}/preprocessing_manifest.json'
with open(MANIFEST_PATH, 'w') as fh:
    json.dump(manifest, fh, indent=2)

print(json.dumps(manifest, indent=2))
print(f'\n✅ Preprocessing complete. Manifest saved to:\n   {MANIFEST_PATH}')
print('\nNext: open Notebook 2 — Architecture, Training & Evaluation')